# RNN Spoken Digit Classifier — Training Notebook
## Tasks A and B1

This notebook trains two models on the **Free Spoken Digit Dataset (FSDD)**:

| Task | Architecture | N_MFCC | Hidden | Constraint |
|------|-------------|--------|--------|------------|
| A    | GRU         | 40     | 128    | None       |
| B1   | MGU (custom)| 13     | 50     | 36 kB/layer|

Task B1 uses **knowledge distillation** from the Task A teacher, training a
memory-constrained student that fits entirely within 36 kB per layer of on-chip SRAM.

## Cell 1 — Environment Setup

Clone the Free Spoken Digit Dataset from GitHub and install the two audio
processing libraries that are not pre-installed in Colab:
- **librosa**: MFCC extraction and delta computation
- **scikit-learn**: stratified train/val/test split

Run this cell once at the start of every Colab session.

In [ ]:
!git clone https://github.com/Jakobovski/free-spoken-digit-dataset.git
!pip install -q librosa scikit-learn

## Cell 2 — Upload and Extract Project Files

Upload your local project zip file (containing `main.py`, `task_b1_constrained.py`,
`config.py`, `diagnose_b1.py`, and the `utils/` folder).

The extraction handles **Windows-style backslash paths** that appear in zips
created on Windows — these would otherwise be treated as a single flat filename
instead of a nested directory structure.

In [ ]:
from google.colab import files
uploaded = files.upload()

import zipfile, os, shutil

with zipfile.ZipFile(list(uploaded.keys())[0], "r") as z:
    z.extractall("/content/temp_extract")

for f in os.listdir("/content/temp_extract"):
    if '\\' in f:
        parts = f.split('\\')
        dest_dir = os.path.join("/content/project", *parts[:-1])
        os.makedirs(dest_dir, exist_ok=True)
        shutil.move(
            os.path.join("/content/temp_extract", f),
            os.path.join(dest_dir, parts[-1])
        )
    else:
        shutil.move(
            os.path.join("/content/temp_extract", f),
            f"/content/project/{f}"
        )

shutil.rmtree("/content/temp_extract")
print("Extraction complete.")

## Cell 3 — Verify GPU Runtime

Training on CPU is too slow for this project (~10 min/epoch for Task B1's
unrolled MGU). This cell asserts that a CUDA GPU is available.

If the assertion fails: **Runtime → Change runtime type → Hardware accelerator → GPU**

In [ ]:
import torch
assert torch.cuda.is_available(), "Switch to GPU runtime first (Runtime > Change runtime type)"
print(torch.cuda.get_device_name(0))

## Cell 4 — Verify Extracted File Structure

Walk the `/content/project` directory and print every `.py` file.
Expected output:
```
/content/project/config.py
/content/project/main.py
/content/project/task_b1_constrained.py
/content/project/diagnose_b1.py
/content/project/utils/data_loader.py
/content/project/utils/data_preprocessing.py
/content/project/utils/memory_utils.py
```
If files are missing, re-run Cell 2 with the correct zip.

In [ ]:
import os
for root, dirs, files in os.walk("/content/project"):
    dirs[:] = [d for d in dirs if d != '__pycache__']
    for f in files:
        if f.endswith('.py'):
            print(os.path.join(root, f))

## Cell 5 — Data Pipeline Diagnostic (MUST PASS before training)

Run `diagnose_b1.py` to verify the data pipeline is correct.
This checks three things that previously caused silent training failures:

1. **Normalisation check** — `x_mean` and `x_std` on a batch must be `≈ 0.00` and `≈ 1.00`
2. **Gradient flow** — all layers should have non-zero gradient norms
3. **Loss trajectory** — loss should drop below 2.30 within 10 steps

> **x_mean must be ≈ 0.00 and x_std must be ≈ 1.00 before proceeding.**
> If normalisation is wrong, zero-padded frames corrupt every batch and the
> model cannot learn meaningful temporal patterns.

In [ ]:
%cd /content/project
!python diagnose_b1.py

## Cell 6 — Task A Training (Baseline GRU)

Train the unconstrained **Task A** GRU classifier:
- Architecture: GRU with `hidden=128`, `N_MFCC=40` (feature_dim=120)
- Target: ≥ 96% test accuracy
- Output checkpoint: `best_model.pt`

This checkpoint is required by Task B1, which loads it as the **teacher model**
for knowledge distillation. Run Task A before Task B1.

In [ ]:
!python main.py

## Cell 7 — Task B1 Training (Constrained MGU + Distillation)

Train the memory-constrained **Task B1** student model:
- Architecture: Custom MGU with `hidden=50`, `N_MFCC=13` (feature_dim=39)
- Every layer is individually verified to fit within **36 kB** of SRAM
- Knowledge distillation from the Task A GRU teacher (`best_model.pt`)
- Loss: `0.3 × CE(hard labels) + 0.7 × KL(soft targets, T=3.0)`
- Output checkpoint: `best_model_b1_constrained.pt` (student weights only)

The script will abort with a clear error if `best_model.pt` is missing or if
any layer exceeds the 36 kB constraint.

In [ ]:
!python task_b1_constrained.py

## Cell 8 — Download Checkpoints

Download both trained model checkpoints to your local machine:
- `best_model.pt` — Task A GRU teacher (full-size, unconstrained)
- `best_model_b1_constrained.pt` — Task B1 MGU student (36 kB/layer constraint)

If a checkpoint is missing, the corresponding training cell did not complete
successfully — check the output above for errors.

In [ ]:
from google.colab import files
import os

for ckpt in ["best_model.pt", "best_model_b1_constrained.pt"]:
    path = f"/content/project/{ckpt}"
    if os.path.exists(path):
        files.download(path)
    else:
        print(f"Not found: {ckpt}")